In [ ]:
%load_ext autoreload

In [ ]:
from pathlib import Path
import re

import pandas as pd
import seaborn as sns
import numpy as np
import mne
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

In [ ]:
%autoreload 2

from src.data import add_metadata_features
from src.models import causal4

In [ ]:
max_log_stim_p = -np.log10(0.01)
min_log_p = -np.log10(0.05)

tg_dir = "textgrids"

outdir = "."

timit_epoch_sources = {
    "All": "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-all.h5",
    "Word onset": "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-onsets.h5",
}

electrodes_paths = list(Path("outputs/causal4/find_speech_responsive").glob("*_results.csv"))

epochs_paths = list(Path("outputs/epochs_preprocessed").glob("*.fif"))
A_result_path = Path("outputs/causal4/unify_As/results.csv")
A_decoders_path = Path("outputs/causal4/unify_As/unified_decoders.pt")
all_B_result_paths = list(Path("outputs/causal4/find_Bs").glob("*_results.csv"))

In [ ]:
epochs = {
    re.search(r"(\w+)_epo.fif", str(path)).group(1): mne.read_epochs(path, preload=True, verbose=False)
    for path in epochs_paths
}

In [ ]:
for e in epochs.values():
    e.metadata = add_metadata_features(e.metadata)

In [ ]:
electrode_df = pd.concat([pd.read_csv(path) for path in electrodes_paths]).set_index(["subject", "electrode_idx"])

In [ ]:
A_results = pd.read_csv(A_result_path)
B_results = pd.concat(
    [pd.read_csv(path) for path in all_B_result_paths],
    ignore_index=True,
)

In [ ]:
A_decoders = torch.load(A_decoders_path)

In [ ]:
g = sns.jointplot(data=B_results, x="p_val_min_log", y="stim_control_p_val_min_log")
g.ax_joint.axvline(min_log_p, color="red", linestyle="--")
g.ax_joint.axhline(max_log_stim_p, color="red", linestyle="--")
# fill rect
g.ax_joint.fill_betweenx([max_log_stim_p, g.ax_joint.get_ylim()[1]], 0, min_log_p, color="red", alpha=0.1)
# plot y=x
g.ax_joint.plot(g.ax_joint.get_xlim(), g.ax_joint.get_ylim(), ls="--", c=".3")

In [ ]:
assert set(epochs.keys()) == set(A_decoders["train_scores"].subject.unique())
assert set(epochs.keys()) == set(B_results.subject)
subjects = sorted(A_results.subject.unique())

In [ ]:
study_df = B_results[(B_results["p_val_min_log"] < min_log_p) & (B_results["stim_control_p_val_min_log"] > max_log_stim_p)]
study_df = study_df.sort_values("stim_control_p_val_min")
study_df

In [ ]:
study_df.to_csv(f"{outdir}/C_study_results.csv", index=False)

In [ ]:
# plotter = causal4.Causal4Plotter(
#     epochs=epochs,
#     A_results=A_results,
#     B_results=B_results,
#     A_decoders=A_decoders,
#     electrode_df=electrode_df,
#     textgrid_dir=tg_dir,
#     timit_epoch_sources=timit_epoch_sources,
# )

In [ ]:
# from matplotlib.backends.backend_pdf import PdfPages

# limit = 20
# i = 0
# with PdfPages(f"{outdir}/B_stim_study.pdf") as pdf:
#     for _, row in tqdm(B_results.iterrows(), total=len(B_results)):
#         facetgrids = plotter(row)
#         for fg in facetgrids:
#             if fg is None:
#                 continue
#             fg.tight_layout()
#             fig = fg.fig if hasattr(fg, 'fig') else fg
#             pdf.savefig(fig)
#             plt.close(fig)

#         i += 1
#         if i > limit:
#             break